In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import ExtraTreesRegressor


In [3]:
df = pd.read_csv("cleaned.csv", delimiter=',' ,encoding='latin-1')

In [4]:

import re
import unicodedata
from collections import Counter, defaultdict

import pandas as pd


# Ilmselgelt mitte mudelinimed.
BAD_START = {
    "andere", "sonstige", "sonstiges",
    "auto", "fahrzeug", "diesel", "benzin",
    "tuv", "tuev", "rot",
    "kombi", "cabrio", "kleinwagen",
    "sportwagen", "sportwagon",
    "sommerfahrzeug", "autoverkauf",
    "gebrauchte", "gebraucht",
    "wegen", "mit", "ich", "meine", "alles",
    "in", "guenstiger", "zuverlaessiges",
    "schmucker", "sparsamer", "vollaustattung",
    "sonderangebot", "selespeed", "jts", "jtd",
    "sportliches",
}

# Mootori-, varustuse- ja muud lisatähised,
# mis ei peaks eraldi mudelikategooriat tekitama.
BAD_WORDS = BAD_START | {
    "jtdm", "tdi", "tsi", "cdi", "hdi", "dci",
    "fsi", "gdi", "ts", "twin", "spark",
    "kw", "ps", "hp", "aut", "automatik",
    "automatic", "leder", "junior",
    "quadrifogli", "selespeed",
    "sport", "line", "comfortline",
    "trendline", "highline", "sline",
    "avant",
    "grau", "rot", "schwarz", "weiss",
    "silber", "blau", "gruen", "braun",
    "gelb", "beige", "metallic",
}

COLORS = {
    "grau", "rot", "schwarz", "weiss",
    "silber", "blau", "gruen", "braun",
    "gelb", "beige", "metallic",
}


def normalize_text(value):
    """Muudab teksti ühtsele kujule."""
    if pd.isna(value):
        return ""

    value = str(value).casefold()
    value = unicodedata.normalize("NFKD", value)
    value = "".join(
        char for char in value
        if not unicodedata.combining(char)
    )
    value = value.replace("ß", "ss")
    value = re.sub(r"[^a-z0-9]+", " ", value)

    return " ".join(value.split())


def compact(value):
    """Eemaldab tühikud, et kirjapildi variante võrrelda."""
    return re.sub(r"[^a-z0-9]", "", value)


def format_model(value):
    """Ühtlustab mudelinime kirjapildi."""
    value = normalize_text(value)

    # C 200 -> C200
    value = re.sub(
        r"\b([a-z])\s+(\d+)\b",
        r"\1\2",
        value
    )

    # 328 i -> 328i
    value = re.sub(
        r"\b(\d+)\s+([a-z])\b",
        r"\1\2",
        value
    )

    return value


def strip_brand(model, brand):
    """Eemaldab mudelinime ette sattunud automargi."""
    model_parts = model.split()
    brand_parts = brand.split()

    # Näiteks: alfa romeo 147 -> 147
    if brand_parts and model_parts[:len(brand_parts)] == brand_parts:
        return " ".join(model_parts[len(brand_parts):])

    # Näiteks: alfa 156sw -> 156sw
    if brand_parts and model_parts and model_parts[0] == brand_parts[0]:
        return " ".join(model_parts[1:])

    # Näiteks: alfaromeogtv -> gtv
    model_key = compact(model)
    brand_key = compact(brand)

    if brand_key and model_key.startswith(brand_key):
        return model_key[len(brand_key):]

    return model


def is_model_code(key):
    """Tunneb ära tüüpilise mudelikoodi kuju."""
    return bool(
        re.fullmatch(r"\d{2,3}", key)
        or re.fullmatch(r"\d{1,3}[a-z]", key)
        or re.fullmatch(r"[a-z]\d{1,3}[a-z]?", key)
    )


def is_bad_candidate(display):
    """Kontrollib, kas väärtus sobib mudelinimeks."""
    if not display:
        return True

    tokens = display.split()

    if tokens[0] in BAD_START:
        return True

    if len(tokens) > 2:
        return True

    if any(token in BAD_WORDS for token in tokens):
        return True

    if any(
        re.fullmatch(r"\d+(kw|ps|hp)", token)
        for token in tokens
    ):
        return True

    return False


def clean_car_models(
    df,
    brand_col="brand",
    model_col="clean_model",
    output_col="clean_model_2",
    min_frequency=15
):
    """
    Puhastab automaatselt kõigi automarkide mudelinimed.

    Tagastab:
        cleaned_df: DataFrame koos puhastatud mudelite veeruga.
        unknowns: tabel väärtustest, mida ei suudetud tuvastada.
    """

    prepared = []
    frequencies = Counter()
    forms = defaultdict(Counter)

    # 1. Normaliseerime kõik väärtused.
    for brand, model in zip(df[brand_col], df[model_col]):

        brand_norm = normalize_text(brand)
        model_norm = normalize_text(model)

        model_norm = strip_brand(model_norm, brand_norm)

        brand_key = compact(brand_norm)
        model_key = compact(model_norm)
        display = format_model(model_norm)

        prepared.append((brand_key, model_key, display))

        if model_key:
            frequencies[(brand_key, model_key)] += 1
            forms[(brand_key, model_key)][display] += 1

    # 2. Koostame iga margi mudelikataloogi.
    candidates = defaultdict(dict)

    for (brand, key), count in frequencies.items():

        display = forms[(brand, key)].most_common(1)[0][0]
        tokens = display.split()

        # Säilitame haruldased mudelinimed,
        # millele järgneb ainult värvinimetus.
        if (
            len(tokens) == 2
            and tokens[1] in COLORS
            and re.fullmatch(r"[a-z]{4,}", tokens[0])
            and tokens[0] not in BAD_START
        ):
            candidates[brand].setdefault(
                compact(tokens[0]), tokens[0]
            )

        if is_bad_candidate(display):
            continue

        # Mudel lisatakse, kui see on piisavalt sage
        # või näeb välja nagu tüüpiline mudelikood.
        is_single_word_name = (
            len(tokens) == 1
            and re.fullmatch(r"[a-z]{4,}", tokens[0])
        )

        if (
            count >= min_frequency
            or is_model_code(key)
            or is_single_word_name
        ):
            candidates[brand][key] = display

    # 3. Sobitame kõik ülejäänud väärtused kataloogiga.
    cleaned_models = []

    for brand, key, display in prepared:

        if not key or not display:
            cleaned_models.append("unknown")
            continue

        # Ilmselge kuulutuse tekst.
        if display.split()[0] in BAD_START:
            cleaned_models.append("unknown")
            continue

        brand_candidates = candidates[brand]

        # Täpne vaste.
        if key in brand_candidates:
            cleaned_models.append(brand_candidates[key])
            continue

        # Tüüpiline haruldane mudelikood.
        if is_model_code(key):
            cleaned_models.append(display)
            continue

        # Pikemad mudelinimed enne lühemaid,
        # et näiteks GTV ei läheks GT alla.
        matched = None

        for candidate in sorted(
            brand_candidates,
            key=len,
            reverse=True
        ):
            if len(candidate) < 2:
                continue

            if not key.startswith(candidate):
                continue

            suffix = key[len(candidate):]

            # Väldime näiteks A4 sobitamist A40 külge.
            if (
                suffix
                and candidate[-1].isdigit()
                and suffix[0].isdigit()
            ):
                continue

            matched = brand_candidates[candidate]
            break

        cleaned_models.append(
            matched if matched is not None else "unknown"
        )

    # 4. Säilitame originaalandmed ja lisame uue veeru.
    result = df.copy()
    result[output_col] = cleaned_models

    # 5. Koostame ülevaate väärtustest, mida ei tuvastatud.
    unknowns = (
        result[result[output_col] == "unknown"]
        .groupby([brand_col, model_col], dropna=False)
        .size()
        .rename("count")
        .reset_index()
        .sort_values("count", ascending=False)
    )

    return result, unknowns

df, unknowns = clean_car_models(
    df,
    brand_col="brand",
    model_col="clean_model",
    min_frequency=15
)

print(df["clean_model_2"].value_counts())
print(unknowns.head(10))# Loeme iga margi ja mudeli kombinatsiooni esinemissageduse
model_counts = df.groupby(
    ["brand", "clean_model_2"]
)["clean_model_2"].transform("size")

# Ühendame haruldased mudelid kategooriasse other
rare_models = (
    (model_counts < 3)
    & (df["clean_model_2"] != "unknown")
)

df["model_final"] = df["clean_model_2"]
df.loc[rare_models, "model_final"] = "other"

clean_model_2
golf                   19667
unknown                 9728
a4                      8279
polo                    7972
passat                  7644
                       ...  
gollf                      1
645i                       1
c 300t                     1
spacelight                 1
leistungssteigerung        1
Name: count, Length: 1830, dtype: int64
              brand clean_model  count
711             bmw         NaN   2319
4209     volkswagen         NaN    193
2205  mercedes_benz         NaN    138
2917        peugeot         NaN    134
255            audi         NaN    106
759       chevrolet       spark    105
3590          smart         NaN    100
2759           opel         NaN     98
3545          smart       smart     73
44       alfa_romeo         NaN     72


In [5]:
# Loeme iga margi ja mudeli kombinatsiooni esinemissageduse
model_counts = df.groupby(
    ["brand", "clean_model_2"]
)["clean_model_2"].transform("size")

# Ühendame haruldased mudelid kategooriasse other
rare_models = (
    (model_counts < 3)
    & (df["clean_model_2"] != "unknown")
)

df["model_final"] = df["clean_model_2"]
df.loc[rare_models, "model_final"] = "other"

In [20]:
df = df[df.fuelType != "andere"]
X = df.drop(columns=["price", "seller", "abtest", "name", "clean_model", "clean_model_2", "model", "dateCrawled", "monthOfRegistration", "yearOfRegistration", "listingAge"])
X["carAge"] = 2016 - df.yearOfRegistration
y = df["price"]
X.head()

,vehicleType,gearbox,powerPS,kilometer,fuelType,brand,notRepairedDamage,model_final,carAge
0,suv,automatik,163,125000,diesel,jeep,NaN,grand cherokee,12
1,kleinwagen,manuell,75,150000,benzin,volkswagen,nein,golf,15
2,kleinwagen,manuell,69,90000,diesel,skoda,nein,fabia,8
3,limousine,manuell,102,150000,benzin,bmw,ja,316i,21
4,cabrio,manuell,109,150000,benzin,peugeot,nein,206,12


In [7]:
numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "string", "bool"]
).columns.tolist()

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

Numeric: ['powerPS', 'kilometer', 'carAge']
Categorical: ['vehicleType', 'gearbox', 'fuelType', 'brand', 'notRepairedDamage', 'model_final']


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

In [9]:
X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

In [10]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

In [11]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="unknown"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=20,
        sparse_output=False
    ))
])

In [15]:


categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant",
                              fill_value="unknown")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=20
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", GradientBoostingRegressor(
        random_state=42,
        learning_rate=0.03,
        max_leaf_nodes=15,
        n_estimators=400,
        max_depth=3,
        min_samples_leaf=3))
])

In [16]:
model.fit(X_train_2, y_train_2)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['vehicleType','gearbox','powerPS',...,'notRepairedDamage','model_final', 'carAge']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This sub

In [17]:
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

predictions = model.predict(X_test_2)

mae = mean_absolute_error(y_test_2, predictions)
rmse = np.sqrt(mean_squared_error(y_test_2, predictions))
r2 = r2_score(y_test_2, predictions)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 1548.9499270937856
RMSE: 2768.618993048063
R²: 0.8757807600165295


In [21]:
model2 = Pipeline([
    ("preprocessor", preprocessor),

("Extra Trees", ExtraTreesRegressor(
        n_estimators=50,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42))
])

In [22]:
model2.fit(X_train_2, y_train_2)
predictions2 = model2.predict(X_test_2)

mae = mean_absolute_error(y_test_2, predictions2)
rmse = np.sqrt(mean_squared_error(y_test_2, predictions2))
r2 = r2_score(y_test_2, predictions2)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 1146.5603085136954
RMSE: 2209.0162935277567
R²: 0.9209211548404862
